# CASSCF baseline, DMRG, Riemannian polish, RPA and GAD on the CASSCF-paper systems — SU(2) spin-adapted layer

Systems: the three molecules of arXiv:2604.13753 (Grazioli, Hu, Nottoli, Lipparini, Cancès,
*Critical point search and linear response theory … Part II. CASSCF*): water CAS(8,6),
formaldehyde CAS(4,3), ethylene CAS(2,2); singlets, 6-31G, fixture geometries, Hartree-Fock
canonical orbitals.

Per system:

1. CASSCF-equivalent: joint TT ⊕ orbital ground state on the pinned CAS grid, then joint RPA at
   that point.
2. DMRG (two-site MALS) to the full-space ground state at truncation tolerance `DMRG_EPS` under
   the reduced-multiplet cap `DMRG_CAP`, on the untyped space and on the point-group-typed
   space.
3. Riemannian CG polish of both DMRG states to `‖grad‖ < GRADTOL`.
4. Signed RPA at each polished state; one table against the paper, `f_W` beside every mode.
5. GAD from each polished state.

No parameter search, no tuning to reference values, no orbital rotation other than the CASSCF.
The ground state is real throughout. Notebook 04 is the abelian layer, 05 the SU(2) layer, from
one generator (`paper_cells` in `mknotebooks.py`); they share `notebooks/cache/`.

In [1]:
# Use the notebook environment (see notebooks/Project.toml) regardless of which kernel started us.
using Pkg
Pkg.activate(@__DIR__)

  Activating 

project at `~/Dropbox/Workplace/RustPackages/qntt/notebooks`


In [2]:
using QNTT, LinearAlgebra, Printf, Serialization

# RAYON_THREADS is set by the cluster job; 0 lets rayon choose.
QNTT.init(parse(Int, get(ENV, "RAYON_THREADS", "0")))
@printf("qntt %s, %d threads, %s layer\n", QNTT.version(), QNTT.nthreads(), "su2")

# Where the integral fixtures live, and a loader for the pyscf-exported `meta/h1/h2` format
# (used by the LiH and N2 fixtures that are not FCIDUMP files).
const ROOT = normpath(joinpath(@__DIR__, ".."))
fixture(parts...) = joinpath(ROOT, "testdata", parts...)

"""
Load a fixture directory: a keyed `meta.txt` plus the FCIDUMP its `dump_file` key names.

`e_ref` is *whatever reference the fixture was generated against*, and the meta says which by the
key it uses — `e_fci` where the determinant count allowed one, `e_ccsd_t` otherwise. It is never a
generic slot whose meaning varies per fixture (two fixtures used to record `e_ref == e_hf`, from a
skipped FCI silently falling back to the SCF energy).

These directories used to ship `h1.dat`/`h2.dat`: a dense positional dump with no write threshold,
which carried entries down to ~1e-22 where an FCIDUMP is written at 1e-15 — the same molecule with
materially different content (66% of `h2.dat` and 17.5% of the assembled operator at N₂/cc-pVDZ).
One format, so the question does not arise.
"""
function load_fixture(dir)
    meta = Dict{String,String}()
    for line in eachline(joinpath(dir, "meta.txt"))
        line = strip(line)
        (isempty(line) || startswith(line, "#")) && continue
        parts = split(line, limit = 2)
        length(parts) == 2 && (meta[parts[1]] = strip(parts[2]))
    end
    getf(k) = parse(Float64, meta[k])
    d      = parse(Int, meta["norb"])
    nelec  = parse(Int, meta["nelec"])
    ms2    = parse(Int, get(meta, "ms2", "0"))
    e_nuc  = getf("e_nuc")
    e_hf   = haskey(meta, "e_hf") ? getf("e_hf") : NaN
    refkey = findfirst(k -> haskey(meta, k), ["e_fci", "e_ccsd_t", "e_ccsd"])
    e_ref  = refkey === nothing ? NaN : getf(["e_fci", "e_ccsd_t", "e_ccsd"][refkey])

    ints = QNTT.read_fcidump(joinpath(dir, meta["dump_file"]))
    t, v = QNTT.onebody(ints), QNTT.twobody(ints)
    @assert size(t) == (d, d) "meta norb=$d disagrees with $(meta["dump_file"])"
    return (; t, v, d, nelec, ms2, e_nuc, e_hf, e_ref)
end

# --- the state cache ---------------------------------------------------------------------------
# The ground-state pipeline is deterministic and expensive (~2 minutes for water 6-31G at M = 64),
# while every downstream experiment repeats it verbatim. So: build once, save, and reuse. Cold on the first execution, warm in milliseconds
# thereafter.
#
# The cache is a local file, NOT an archival format: it records the build that wrote it and refuses
# anything else, and there is no migration path. If `load` ever complains, delete the file and let
# the cold path run again.
#
# It keys on the physical setup only (fixture, 2S, M and a tag). CHANGING A SOLVER KNOB DOES NOT
# INVALIDATE IT — delete `notebooks/cache/` by hand after editing the schedule below, or you will
# be reading a state built with the old one. The space is re-checked on load, so a change that
# alters the space is caught; one that only alters the path to the state is not.
const CACHE_DIR = joinpath(@__DIR__, "cache")
cache_file(tag) = joinpath(CACHE_DIR, tag * ".tt")

"""
    cached(T, L, tag, space, build) -> (x, :warm|:cold, seconds)

Return the cached state `tag`, or build it with `build()` and save it. The loaded state's space
must equal `space`, so a cache left over from a different physical setup is an error rather than a
silent wrong answer.
"""
function cached(::Type{T}, ::Type{L}, tag, space, build) where {T,L}
    path = cache_file(tag)
    if isfile(path)
        t = @elapsed x = QNTT.load(T, L, path)
        QNTT.space(x) == space ||
            error("cache $path holds $(QNTT.space(x)), but this run wants $space; delete it")
        return (x, :warm, t)
    end
    t = @elapsed x = build()
    mkpath(CACHE_DIR)
    QNTT.save(x, path)
    return (x, :cold, t)
end

report_cache(tag, how, t) = how === :warm ?
    @printf("cache HIT  %s — loaded in %.3f s (the pipeline below was skipped)\n", tag, t) :
    @printf("cache MISS %s — built in %.1f s and saved for next time\n", tag, t)

# --- the systems of arXiv:2604.13753 (Part II, CASSCF) ------------------------------------------
# HF-canonical-MO fixtures with ORBSYM: h2o C2v, hcho C2v, c2h4 D2h. `cas` is the
# (n_core, n_active, n_virt) partition — cores first, virtuals last, as pyscf canonicalizes it.
const SYSTEMS = (
    (dir = "h2o_631g",  mol = "h2o",  label = "water CAS(8,6) / 6-31G",        cas = (1, 6, 6)),
    (dir = "hcho_631g", mol = "hcho", label = "formaldehyde CAS(4,3) / 6-31G", cas = (6, 3, 13)),
    (dir = "c2h4_631g", mol = "c2h4", label = "ethylene CAS(2,2) / 6-31G",     cas = (7, 2, 17)),
)
system(dir) = SYSTEMS[findfirst(s -> s.dir == dir, SYSTEMS)]

# Paper Table II (Hartree): rows ω₁, ω₂; columns SS / SA / LR. The LR column is what the
# CASSCF-equivalent below reproduces (certified in tests/joint_cas at ≤ 5e-7); SS and SA are
# different methods, printed for context only.
const PAPER = Dict(
    "h2o"  => [0.299801 0.279149 0.313274; 0.377632 0.357262 0.388773],
    "hcho" => [0.144960 0.147317 0.138848; 0.335188 0.464643 0.325269],
    "c2h4" => [0.384667 0.388940 0.354318; 0.567533 0.568822 0.562656],
)

"""
Scrape one value from the `6-31g` block of molecule `mol` in `testdata/casscf_paper_references.json`:
a scalar, a flat array, or `nothing` for a JSON `null`. Textual on purpose (no JSON dependency, as
notebook 03 does); every caller asserts on what comes back.
"""
function ref_value(mol, key)
    refs = read(fixture("casscf_paper_references.json"), String)
    function cut(s, pat)
        r = findfirst(pat, s)
        r === nothing && error("$pat not found in the reference file for $mol")
        return s[last(r)+1:end]
    end
    s = lstrip(cut(cut(cut(refs, "\"$mol\""), "\"6-31g\""), "\"$key\":"))
    startswith(s, "null") && return nothing
    num = r"-?\d+\.\d+(?:[eE][-+]?\d+)?"
    if startswith(s, "[")
        body = s[2:findfirst(']', s)-1]
        return [parse(Float64, m.match) for m in eachmatch(num, body)]
    end
    return parse(Float64, match(num, s).match)
end

"""
    cached_value(tag, build) -> (value, :warm|:cold, seconds)

`Serialization`-based sibling of `cached` for the small results (ω lists, energies, info tuples).
Same contract: keyed on the tag alone, so delete `notebooks/cache/` after changing a knob.
"""
function cached_value(tag, build)
    path = joinpath(CACHE_DIR, tag * ".jls")
    if isfile(path)
        t = @elapsed v = deserialize(path)
        return (v, :warm, t)
    end
    t = @elapsed v = build()
    mkpath(CACHE_DIR)
    serialize(path, v)
    return (v, :cold, t)
end

"Nearest entry of `ω` to `g`: (value, 1-based index, ω − g), or `nothing` on an empty spectrum."
function nearest(ω, g)
    isempty(ω) && return nothing
    j = argmin(abs.(ω .- g))
    return (ω[j], j, ω[j] - g)
end
fmt_near(n) = n === nothing ? "      —          " : @sprintf("%.6f [%2d] %+.1e", n[1], n[2], n[3])
"Nearest mode with its Weingarten fraction beside it: `ω [index] deviation  f_W`."
fmt_near(n, f_w) = n === nothing ? @sprintf("%-32s", "—") :
    @sprintf("%.6f [%2d] %+.1e f_W %s", n[1], n[2], n[3], isempty(f_w) ? "  ?  " : @sprintf("%5.2f", f_w[n[2]]))
fmt_fw(f_w, k) = isempty(f_w) ? "—" : join([@sprintf("%.2f", w) for w in f_w[1:min(k, length(f_w))]], "    ")

# `flush(stdout)` after each report: a no-op in Jupyter, and what makes a redirected `nbconvert`
# or script log stream while the long Rust calls run (Julia's I/O loop is blocked inside them).
fmt_ω(ω, k) = isempty(ω) ? "—" : join([@sprintf("%.5f", w) for w in ω[1:min(k, length(ω))]], " ")


const LAYER = "su2"
# Systems to run (h2o: d = 13; hcho: d = 22; c2h4: d = 26).
const RUN = ["h2o_631g", "hcho_631g", "c2h4_631g"]

qntt 0.1.0, 16 threads, su2 layer


3-element Vector{String}:
 "h2o_631g"
 "hcho_631g"
 "c2h4_631g"

## Setup

Reads each FCIDUMP with its point group, screens the symmetry-forbidden entries, builds the
untyped space and the typed space and one operator on each. Prints `E_HF`, `E_ref`, the pyscf
CASSCF energy, the paper's LR values and both construction times.

In [3]:
"Load one paper system: screened integrals, the two spaces, the two operators, the references."
function setup(dir)
    sys = system(dir)
    fx = load_fixture(fixture(dir))
    dump = only(filter(f -> endswith(f, ".FCIDUMP"), readdir(fixture(dir))))
    data = QNTT.read_fcidump(joinpath(fixture(dir), dump); point_group = :auto)
    QNTT.screen_forbidden!(data)                 # MANDATORY before a point-group-typed construction
    pg = QNTT.pointgroup(data)
    nup = (fx.nelec + fx.ms2) ÷ 2
    ndn = fx.nelec - nup
    s = (; sys, fx, data, pg, nup, ndn)
    sp_c1 = QNTT.Su2Space(s.fx.d, s.fx.nelec, 0)
    sp_pg = QNTT.Su2Space(data; two_s = 0, point_group = :inherit)
    t_c1 = @elapsed H_c1 = QNTT.Hamiltonian(sp_c1, data)
    t_pg = @elapsed H_pg = QNTT.Hamiltonian(sp_pg, data)
    casscf_gs = ref_value(sys.mol, "casscf_gs")
    fci_gaps  = ref_value(sys.mol, "fci_gaps")        # `nothing` where no FCI exists (hcho, c2h4)
    return (; s..., sp_c1, sp_pg, H_c1, H_pg, t_c1, t_pg, casscf_gs, fci_gaps)
end

S = Dict(dir => setup(dir) for dir in RUN)
for dir in RUN
    s = S[dir]
    println(s.sys.label)
    @printf("  d = %d, N = %d, %s (target %s)   E_HF = %.9f   E_ref (%s) = %.9f\n",
            s.fx.d, s.fx.nelec, s.pg.kind, s.pg.target, s.fx.e_hf,
            isnan(s.fx.e_ref) ? "none" : (s.fci_gaps === nothing ? "CCSD(T)" : "FCI"), s.fx.e_ref)
    @printf("  pyscf CASSCF   = %.9f      paper LR ω₁, ω₂ = %.6f, %.6f\n",
            s.casscf_gs, PAPER[s.sys.mol][1, 3], PAPER[s.sys.mol][2, 3])
    @printf("  untyped space  %s   built in %6.1f s\n", string(s.sp_c1), s.t_c1)
    @printf("  typed space    %s   built in %6.1f s  (%.1f× the untyped build)\n",
            string(s.sp_pg), s.t_pg, s.t_pg / s.t_c1)
    flush(stdout)
end

water CAS(8,6) / 6-31G


  d = 13, N = 10, C2v (target 0)   E_HF = -75.980157922   E_ref (FCI) = -76.121285071
  pyscf CASSCF   = -76.039557117      paper LR ω₁, ω₂ = 0.313274, 0.388773


  untyped space  Su2Space(d=13, N=10, 2S=0)   built in    0.9 s
  typed space    Su2Space(d=13, N=10, 2S=0, C2v, target=A1)   built in    3.7 s  (4.2× the untyped build)


formaldehyde CAS(4,3) / 6-31G
  d = 22, N = 16, C2v (target 0)   E_HF = -113.806879119   E_ref (CCSD(T)) = -114.043672444
  pyscf CASSCF   = -113.849424553      paper LR ω₁, ω₂ = 0.138848, 0.325269
  untyped space  Su2Space(d=22, N=16, 2S=0)   built in   20.5 s
  typed space    Su2Space(d=22, N=16, 2S=0, C2v, target=A1)   built in  100.7 s  (4.9× the untyped build)


ethylene CAS(2,2) / 6-31G
  d = 26, N = 16, D2h (target 0)   E_HF = -78.004104910   E_ref (CCSD(T)) = -78.218487193
  pyscf CASSCF   = -78.034920730      paper LR ω₁, ω₂ = 0.354318, 0.562656
  untyped space  Su2Space(d=26, N=16, 2S=0)   built in   25.1 s
  typed space    Su2Space(d=26, N=16, 2S=0, D2h, target=Ag)   built in  263.7 s  (10.5× the untyped build)


## 1. CASSCF-equivalent

`cas_reference_state` builds the pinned CAS(n, m) grid (core and virtual bonds carry one sector
at rank 1). `casscf` runs the joint TT ⊕ orbital solver from the fixture orbitals with the grid
held fixed. `joint_rpa` returns the symplectic eigenvalues of the joint Hessian at the converged
point and `f_W` per mode. The energy is compared with pyscf's CASSCF (`casscf_gs`), the two
lowest ω with the paper's LR column. ω are printed only under `converged && !rank_limited`.
`CASSCF_START = :cas` starts from the `*_cas` fixture twin (pyscf CASSCF orbitals) instead.

In [4]:
const CASSCF_START    = :hf      # :hf = from the fixture (HF) orbitals; :cas = from the `*_cas` twin
const CASSCF_ORB_ITER = 40       # orbital iterations
const CASSCF_ORB_GRAD = 1e-6     # orbital-gradient gate
const CASSCF_TT_GRAD  = 1e-6     # TT-leg gradient tolerance
const CASSCF_SEED     = 0xCA5F0
const SHOW_K          = 6        # how many ω to print per spectrum

function casscf_baseline(s)
    sys = s.sys
    if CASSCF_START === :hf
        data, H = s.data, s.H_c1
    else
        dir_cas = fixture(sys.dir * "_cas")
        dump = only(filter(f -> endswith(f, ".FCIDUMP"), readdir(dir_cas)))
        data = QNTT.read_fcidump(joinpath(dir_cas, dump))          # C1 on purpose (see its meta.txt)
        H = QNTT.Hamiltonian(QNTT.Su2Space(data; two_s = 0, point_group = nothing), data)
    end
    t0 = time()
    λ0, x0 = QNTT.cas_reference_state(H, sys.cas; m_active = 0, seed = CASSCF_SEED)
    λ, x, u, H_rot, info = QNTT.casscf(data, x0, sys.cas;
                                       orb_max_iter = CASSCF_ORB_ITER, orb_grad = CASSCF_ORB_GRAD,
                                       tt_grad_tol = CASSCF_TT_GRAD, preserve_grid = true)
    t_scf = time() - t0
    e0 = QNTT.e0(data)
    certified = info.converged && !info.rank_limited          # the publication licence
    if certified
        ω, resp = QNTT.joint_rpa(H_rot, x, sys.cas)
    else
        ω, resp = Float64[], (status = :not_run, n_eff = info.n_eff, n_modes = 0, f_w = Float64[],
                              message = "the SCF did not certify — no ω published")
    end
    return (; E = λ + e0, E_start = λ0 + e0, converged = certified, iters = info.iters,
              refused = info.refused_reason, n_eff = info.n_eff,
              g_orb = isempty(info.g_orb_kept) ? NaN : last(info.g_orb_kept),
              ω, f_w = resp.f_w, status = resp.status, n_modes = resp.n_modes, message = resp.message,
              ranks = QNTT.ranks(x), t_scf, t_all = time() - t0)
end

B = Dict{String,Any}()
for dir in RUN
    s = S[dir]
    tag = @sprintf("%s_%s_casscf_%s", dir, LAYER, CASSCF_START)
    B[dir], how, t = cached_value(tag, () -> casscf_baseline(s))
    report_cache(tag, how, t); flush(stdout)
    b = B[dir]
    println(s.sys.label)
    @printf("  start (fixed orbitals)  E = %.9f\n", b.E_start)
    @printf("  CASSCF-equivalent       E = %.9f   Δ vs pyscf CASSCF = %+.2e   %d orbital iterations, \
              ‖g_orb‖ = %.2e, %.1f s\n", b.E, b.E - s.casscf_gs, b.iters, b.g_orb, b.t_scf)
    if !b.converged
        @printf("  DID NOT CERTIFY — no ω published. Refusal: %s\n", something(b.refused, "?"))
        continue
    end
    @printf("  joint RPA: %s, n_eff = %d non-redundant orbital pairs, %d modes, %.1f s total\n",
            b.status, b.n_eff, b.n_modes, b.t_all)
    b.status in (:stable, :resonant) || (println("  ", b.message); continue)
    b.status === :resonant && println("  RESONANT joint point: negative entries below are imaginary pairs (ω = i√|ω²|), not excitations")
    @printf("  ω[1:%d]   = %s\n", SHOW_K, fmt_ω(b.ω, SHOW_K))
    @printf("  f_W[1:%d] = %s\n", SHOW_K, fmt_fw(b.f_w, SHOW_K))
    for j in 1:2
        lr = PAPER[s.sys.mol][j, 3]
        @printf("  paper LR ω%d = %.6f   nearest mode: %s\n", j, lr, fmt_near(nearest(b.ω, lr), b.f_w))
    end
end

cache HIT  h2o_631g_su2_casscf_hf — loaded in 0.109 s (the pipeline below was skipped)


water CAS(8,6) / 6-31G
  start (fixed orbitals)  E = -75.994272742
  CASSCF-equivalent       E = -76.039557119   Δ vs pyscf CASSCF = -1.11e-09   10 orbital iterations, ‖g_orb‖ = 1.22e-06, 52.9 s


  joint RPA: stable, n_eff = 48 non-redundant orbital pairs, 152 modes, 188.1 s total
  ω[1:6]   = 0.31327 0.38877 0.41295 0.49290 0.54944 0.68198


  f_W[1:6] = 0.10    0.06    0.03    0.02    0.01    0.02
  paper LR ω1 = 0.313274   nearest mode: 0.313274 [ 1] +1.5e-07 f_W  0.10


  paper LR ω2 = 0.388773   nearest mode: 0.388773 [ 2] +3.5e-07 f_W  0.06
cache HIT  hcho_631g_su2_casscf_hf — loaded in 0.001 s (the pipeline below was skipped)


formaldehyde CAS(4,3) / 6-31G
  start (fixed orbitals)  E = -113.834785547
  CASSCF-equivalent       E = -113.847644511   Δ vs pyscf CASSCF = +1.78e-03   8 orbital iterations, ‖g_orb‖ = 6.68e-06, 225.7 s
  joint RPA: resonant, n_eff = 135 non-redundant orbital pairs, 140 modes, 1526.4 s total
  RESONANT joint point: negative entries below are imaginary pairs (ω = i√|ω²|), not excitations
  ω[1:6]   = -2.08294 -1.02912 0.13748 0.32044 0.39396 0.39399
  f_W[1:6] = NaN    NaN    0.16    0.09    0.06    0.13
  paper LR ω1 = 0.138848   nearest mode: 0.137479 [ 3] -1.4e-03 f_W  0.16
  paper LR ω2 = 0.325269   nearest mode: 0.320439 [ 4] -4.8e-03 f_W  0.09
cache HIT  c2h4_631g_su2_casscf_hf — loaded in 0.001 s (the pipeline below was skipped)


ethylene CAS(2,2) / 6-31G
  start (fixed orbitals)  E = -78.026170981
  CASSCF-equivalent       E = -78.034920730   Δ vs pyscf CASSCF = -8.24e-11   7 orbital iterations, ‖g_orb‖ = 2.38e-07, 293.2 s
  joint RPA: stable, n_eff = 167 non-redundant orbital pairs, 169 modes, 3325.7 s total
  ω[1:6]   = 0.35162 0.35432 0.37810 0.38916 0.39955 0.42805
  f_W[1:6] = 0.01    0.07    0.00    0.01    0.03    0.03
  paper LR ω1 = 0.354318   nearest mode: 0.354318 [ 2] +1.8e-07 f_W  0.07
  paper LR ω2 = 0.562656   nearest mode: 0.562655 [12] -5.2e-07 f_W  0.01


The spectrum is singlet-only; the paper's two values are expected at indices 1 and 2. At water
the abelian chart has 272 modes, 152 of them singlets.

## 2. DMRG

Start: the closed-shell CSF. Stage A: fixed-`M` ramp `RAMP_BOND_DIMS` with perturbative noise
`RAMP_NOISE`. Stage B: noise-free rank-adaptive MALS at truncation tolerance `DMRG_EPS` under
the reduced-multiplet cap `DMRG_CAP`, re-entered one sweep at a time until the energy changes by less than
`DMRG_STOP`. Same start and schedule on both spaces. Ranks on this layer are the reduced multiplet rank (NOT comparable to an abelian pooled M).

`DMRG_CAP` bounds memory and run time, which grow quickly with the cap; even at the default the
notebook runs for many hours. The cap binds on all three systems, so the results are fixed-rank
results at that cap, with the truncation tolerance printed beside them.

In [5]:
const DMRG_EPS       = 1e-3      # stage-B truncation tolerance
const DMRG_CAP       = 256       # stage-B reduced-multiplet cap (0 = uncapped)
const DMRG_STOP      = 1e-7      # stage-B energy stopping test (Ha), between our own re-entries
const DMRG_MAX_ITER  = 20        # stage-B sweep budget
const RAMP_BOND_DIMS = [16, 32, 64, 128, 128]           # stage A: fixed-M ramp WITH noise …
const RAMP_NOISE     = [1e-3, 1e-4, 1e-5, 1e-6, 0.0]    # … ending at 0.0 (only then is E variational)
const RAMP_EPS       = 1e-8
const RAMP_MAX_ITER  = 6
const RAMP_NOISE_TYPE = :perturbative   # White density-matrix noise, the library default (see below)
const MALS_SEED      = 0x9E3779B9

closed_shell(s, sp) = QNTT.tt_csf([k <= s.fx.nelec ÷ 2 ? :occ : :empty for k in 1:s.fx.d], sp)

# Noise type, measured on water 6-31G from the closed-shell start: random noise leaves the typed
# run 0.035–0.045 Ha above the untyped one (E −76.076 / −76.086 for a 32- / 128-ramp vs −76.1212);
# perturbative noise with the 128-ramp brings both arms to −76.121230.

function dmrg(H, x0, e_nuc, label)
    t_a = @elapsed λa, xa, _ = QNTT.mals(H, x0; bond_dims = RAMP_BOND_DIMS, noise = RAMP_NOISE,
                                         noise_type = RAMP_NOISE_TYPE, eps = RAMP_EPS,
                                         max_iter = RAMP_MAX_ITER, seed = MALS_SEED)
    @printf("  %-8s stage A (M = %d, noisy)     E = %.9f   %7.1f s\n",
            label, last(RAMP_BOND_DIMS), λa + e_nuc, t_a)
    # `eps` is also the driver's energy stopping test; the stop used here is DMRG_STOP, applied
    # between re-entries of one sweep each.
    t_b = time()
    λb, xb, hist = QNTT.mals(H, xa; bond_dims = [DMRG_CAP], noise = [0.0], eps = DMRG_EPS, max_iter = 1)
    sweeps, dw = length(hist), last(hist).discarded_weight
    while sweeps < DMRG_MAX_ITER
        λn, xb, hist = QNTT.mals(H, xb; bond_dims = [DMRG_CAP], noise = [0.0], eps = DMRG_EPS, max_iter = 1)
        sweeps += length(hist); dw = last(hist).discarded_weight
        moved = abs(λn - λb); λb = λn
        moved < DMRG_STOP && break
    end
    @printf("  %-8s stage B (eps = %.0e, cap %d) E = %.9f   %7.1f s   %d sweeps, discarded weight %.2e, max bond %d%s\n",
            label, DMRG_EPS, DMRG_CAP, λb + e_nuc, time() - t_b, sweeps, dw, QNTT.max_bond_rank(xb),
            DMRG_CAP > 0 && QNTT.max_bond_rank(xb) >= DMRG_CAP ? "  (cap reached)" : "")
    flush(stdout)
    return xb
end

X = Dict{String,Any}()        # X[dir] = (c1 = state on the untyped space, pg = state on the typed space)
for dir in RUN
    s = S[dir]
    println(s.sys.label)
    tag_c1 = @sprintf("%s_%s_dmrg_c1_eps%.0e_cap%d", dir, LAYER, DMRG_EPS, DMRG_CAP)
    tag_pg = @sprintf("%s_%s_dmrg_pg_eps%.0e_cap%d", dir, LAYER, DMRG_EPS, DMRG_CAP)
    x_c1, how, t = cached(Float64, QNTT.SU2, tag_c1, s.sp_c1,
                          () -> dmrg(s.H_c1, closed_shell(s, s.sp_c1), s.fx.e_nuc, "untyped"))
    report_cache(tag_c1, how, t); flush(stdout)
    x_pg, how, t = cached(Float64, QNTT.SU2, tag_pg, s.sp_pg,
                          () -> dmrg(s.H_pg, closed_shell(s, s.sp_pg), s.fx.e_nuc, "typed"))
    report_cache(tag_pg, how, t); flush(stdout)
    X[dir] = (c1 = x_c1, pg = x_pg)
    for (label, H, x) in (("untyped", s.H_c1, x_c1), ("typed", s.H_pg, x_pg))
        E = QNTT.rayleigh_quotient(H, x) + s.fx.e_nuc
        @printf("  %-8s E = %.9f   Δ vs CASSCF %+.4e   Δ vs E_ref %+.4e   pooled ranks %s   max sector rank %d\n",
                label, E, E - s.casscf_gs, E - s.fx.e_ref, string([sum(g) for g in QNTT.ranks(x)]),
                QNTT.max_sector_rank(x))
    end
end

water CAS(8,6) / 6-31G


cache HIT  h2o_631g_su2_dmrg_c1_eps1e-03_cap256 — loaded in 0.003 s (the pipeline below was skipped)


cache HIT  h2o_631g_su2_dmrg_pg_eps1e-03_cap256 — loaded in 0.002 s (the pipeline below was skipped)


  untyped  E = -76.121256240   Δ vs CASSCF -8.1699e-02   Δ vs E_ref +2.8831e-05   pooled ranks [1, 3, 7, 19, 49, 143, 256, 256, 196, 104, 31, 10, 3, 1]   max sector rank 63
  typed    E = -76.121257358   Δ vs CASSCF -8.1700e-02   Δ vs E_ref +2.7713e-05   pooled ranks [1, 3, 7, 19, 50, 144, 256, 256, 196, 104, 31, 10, 3, 1]   max sector rank 20


formaldehyde CAS(4,3) / 6-31G
cache HIT  hcho_631g_su2_dmrg_c1_eps1e-03_cap256 — loaded in 0.007 s (the pipeline below was skipped)


cache HIT  hcho_631g_su2_dmrg_pg_eps1e-03_cap256 — loaded in 0.004 s (the pipeline below was skipped)


  untyped  E = -114.041401072   Δ vs CASSCF -1.9198e-01   Δ vs E_ref +2.2714e-03   pooled ranks [1, 3, 5, 12, 24, 59, 133, 256, 256, 256, 256, 256, 256, 256, 255, 217, 163, 118, 67, 30, 10, 3, 1]   max sector rank 113
  typed    E = -114.041295818   Δ vs CASSCF -1.9187e-01   Δ vs E_ref +2.3766e-03   pooled ranks [1, 3, 5, 12, 24, 59, 132, 256, 256, 256, 256, 256, 256, 256, 254, 215, 162, 117, 67, 30, 10, 3, 1]   max sector rank 41


ethylene CAS(2,2) / 6-31G
cache HIT  c2h4_631g_su2_dmrg_c1_eps1e-03_cap256 — loaded in 0.013 s (the pipeline below was skipped)


cache HIT  c2h4_631g_su2_dmrg_pg_eps1e-03_cap256 — loaded in 0.008 s (the pipeline below was skipped)


  untyped  E = -78.214269726   Δ vs CASSCF -1.7935e-01   Δ vs E_ref +4.2175e-03   pooled ranks [1, 3, 5, 13, 26, 62, 133, 202, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 164, 145, 112, 84, 49, 23, 10, 3, 1]   max sector rank 103
  typed    E = -78.214187078   Δ vs CASSCF -1.7927e-01   Δ vs E_ref +4.3001e-03   pooled ranks [1, 3, 5, 13, 26, 61, 129, 196, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 163, 144, 112, 83, 47, 22, 10, 3, 1]   max sector rank 18


## 3. Riemannian polish

One `round!` at `TRIM_EPS` (relative) first, then `polish` re-enters the Riemannian CG until
`‖grad‖ < GRADTOL` or `CG_RESTARTS` is exhausted, then a second trim and polish. Prints `E`,
`‖grad‖` with whether the gate or the budget stopped it, `σ_min` and the pooled ranks.

Stop the polish tight: on water at `DMRG_EPS = 1e-2`, a state stopped at `‖grad‖ < 1e-6` still
shows negative witnesses in both `pd_probe` blocks, and one carried to `1e-8` has none.

In [6]:
const GRADTOL     = 1e-7    # criticality gate
const CG_MAX_ITER = 200
const CG_GRADTOL  = 1e-11   # inner CG stop
const CG_RESTARTS = 4
const TRIM_EPS    = 1e-6
const TRIM_PASSES = 1

"Riemannian CG to criticality: restart until ‖grad‖ < GRADTOL, or give up loudly."
function polish(H, x, λ_scale)
    inner_eps = GRADTOL / max(abs(λ_scale), 1)
    _, y = QNTT.riemannian(H, x; cg = true, eps = inner_eps, max_iter = CG_MAX_ITER, gradtol = CG_GRADTOL)
    ncg = 1
    for _ in 1:CG_RESTARTS
        QNTT.grad_norm(H, y) < GRADTOL && break
        _, y = QNTT.riemannian(H, y; cg = true, eps = inner_eps, max_iter = CG_MAX_ITER, gradtol = CG_GRADTOL)
        ncg += 1
    end
    return (y / norm(y), ncg)
end

function polish_and_trim(H, x, e_nuc, label)
    # Trim first: near-zero Schmidt directions put ~1/σ_min into the Weingarten curvature and
    # slow the CG; removing them before the polish is a conditioning step, not an accuracy one.
    QNTT.round!(x; eps = TRIM_EPS, mode = :rel)
    x = x / norm(x)
    t_p = @elapsed y, ncg = polish(H, x, QNTT.rayleigh_quotient(H, x))
    g = QNTT.grad_norm(H, y)
    @printf("  %-8s CG polish (%2d calls)   E = %.9f   ‖grad‖ = %.2e (%s)   %7.1f s\n",
            label, ncg, QNTT.rayleigh_quotient(H, y) + e_nuc, g,
            g < GRADTOL ? "at gate" : "budget exhausted above GRADTOL", t_p)
    for pass in 1:TRIM_PASSES
        t_t = @elapsed begin
            QNTT.round!(y; eps = TRIM_EPS, mode = :rel)
            y = y / norm(y)
            y, _ = polish(H, y, QNTT.rayleigh_quotient(H, y))
        end
        @printf("  %-8s trim pass %d (eps = %.0e)  E = %.9f   ‖grad‖ = %.2e   σ_min = %.2e   %7.1f s\n",
                label, pass, TRIM_EPS, QNTT.rayleigh_quotient(H, y) + e_nuc, QNTT.grad_norm(H, y),
                QNTT.sigma_min(y), t_t)
    end
    flush(stdout)
    return y
end

P = Dict{String,Any}()        # polished states, same shape as X
for dir in RUN
    s = S[dir]
    println(s.sys.label)
    tag_c1 = @sprintf("%s_%s_dmrg_c1_eps%.0e_cap%d_polished", dir, LAYER, DMRG_EPS, DMRG_CAP)
    tag_pg = @sprintf("%s_%s_dmrg_pg_eps%.0e_cap%d_polished", dir, LAYER, DMRG_EPS, DMRG_CAP)
    y_c1, how, t = cached(Float64, QNTT.SU2, tag_c1, s.sp_c1,
                          () -> polish_and_trim(s.H_c1, X[dir].c1, s.fx.e_nuc, "untyped"))
    report_cache(tag_c1, how, t); flush(stdout)
    y_pg, how, t = cached(Float64, QNTT.SU2, tag_pg, s.sp_pg,
                          () -> polish_and_trim(s.H_pg, X[dir].pg, s.fx.e_nuc, "typed"))
    report_cache(tag_pg, how, t); flush(stdout)
    P[dir] = (c1 = y_c1, pg = y_pg)
    for (label, H, y) in (("untyped", s.H_c1, y_c1), ("typed", s.H_pg, y_pg))
        g = QNTT.grad_norm(H, y)
        @printf("  %-8s E = %.9f   ‖grad‖ = %.2e (%s)   σ_min = %.2e   pooled ranks %s\n",
                label, QNTT.rayleigh_quotient(H, y) + s.fx.e_nuc, g,
                g < GRADTOL ? "critical" : "NOT critical at GRADTOL", QNTT.sigma_min(y),
                string([sum(g) for g in QNTT.ranks(y)]))
    end
end

water CAS(8,6) / 6-31G
cache HIT  h2o_631g_su2_dmrg_c1_eps1e-03_cap256_polished — loaded in 0.003 s (the pipeline below was skipped)


cache HIT  h2o_631g_su2_dmrg_pg_eps1e-03_cap256_polished — loaded in 0.003 s (the pipeline below was skipped)


  untyped  E = -76.121256538   ‖grad‖ = 9.15e-08 (critical)   σ_min = 4.11e-05   pooled ranks [1, 3, 7, 19, 49, 143, 256, 256, 196, 104, 31, 10, 3, 1]
  typed    E = -76.121257687   ‖grad‖ = 8.92e-08 (critical)   σ_min = 3.96e-05   pooled ranks [1, 3, 7, 19, 50, 144, 256, 256, 196, 104, 31, 10, 3, 1]


formaldehyde CAS(4,3) / 6-31G
cache HIT  hcho_631g_su2_dmrg_c1_eps1e-03_cap256_polished — loaded in 0.006 s (the pipeline below was skipped)


cache HIT  hcho_631g_su2_dmrg_pg_eps1e-03_cap256_polished — loaded in 0.005 s (the pipeline below was skipped)


  untyped  E = -114.041442914   ‖grad‖ = 9.84e-08 (critical)   σ_min = 2.61e-05   pooled ranks [1, 3, 5, 12, 24, 59, 133, 256, 256, 256, 256, 256, 256, 256, 255, 217, 163, 118, 67, 30, 10, 3, 1]
  typed    E = -114.041325376   ‖grad‖ = 9.94e-08 (critical)   σ_min = 2.87e-05   pooled ranks [1, 3, 5, 12, 24, 59, 132, 256, 256, 256, 256, 256, 256, 256, 254, 215, 162, 117, 67, 30, 10, 3, 1]


ethylene CAS(2,2) / 6-31G
cache HIT  c2h4_631g_su2_dmrg_c1_eps1e-03_cap256_polished — loaded in 0.014 s (the pipeline below was skipped)


cache HIT  c2h4_631g_su2_dmrg_pg_eps1e-03_cap256_polished — loaded in 0.008 s (the pipeline below was skipped)


  untyped  E = -78.214331986   ‖grad‖ = 9.88e-08 (critical)   σ_min = 2.63e-05   pooled ranks [1, 3, 5, 13, 26, 62, 133, 202, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 164, 145, 112, 84, 49, 23, 10, 3, 1]
  typed    E = -78.214216117   ‖grad‖ = 9.35e-08 (critical)   σ_min = 2.25e-05   pooled ranks [1, 3, 5, 13, 26, 61, 129, 196, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 163, 144, 112, 83, 47, 22, 10, 3, 1]


## 4. Linear response

At each of the three states (CASSCF point, polished untyped DMRG, polished typed DMRG):
`pd_probe(:rr)` and `pd_probe(:ii)` as diagnostics, then `rpa_signed` for the lowest `RPA_K` values.
`rpa_signed` returns signed ω²: positive entries are excitations, negative entries resonances `ω = iκ`, each
with a fresh-apply certificate. It never refuses: an indefinite point comes back solved, with a
nonzero counting offset in `info.report`, its complex modes in `info.complex` and each real mode's
Krein margin in `info.krein` (negative for a de-excitation) — its ω are then not licensed by
`converged` alone. Per mode it returns `f_W = |⟨q,Bq⟩| / |⟨q,Aq⟩|` on `q = X − Y`, with
`A = P_T(H−λ)P_T` and `B` the Weingarten term (`H^rr = A + B`, `H^ii = A − B`); on the `H^ii`
side `f_W > 1` is equivalent to `ω² < 0`.

Response at a point-group-typed reference is irrep-closed: the typed column contains only
totally-symmetric (A1 / Ag) excitations.

In [7]:
const PROBE_BUDGET = 100     # pd_probe Davidson iterations per part
const RPA_K        = 4       # physical ω returned, with multiplicity
const RPA_TOL      = 1e-6
const RPA_MAX_ITER = 1200    # budget in (A, B) apply pairs (pair-equivalents)
const RPA_SEED     = 1

# Linear response at a REAL reference, whatever its stability. `rpa_signed` runs the paired
# Davidson — it never refuses — and returns SIGNED ω²: positive entries are excitations, negative
# ones resonances `ω = iκ`, each certified by fresh applies. Beside every mode it returns the
# Weingarten fraction `f_W = |⟨q,Bq⟩| / |⟨q,Aq⟩|` (see the prose below the cell).
function response(H, x, label)
    t0 = time()
    pr = QNTT.pd_probe(H, x; part = :rr, budget = PROBE_BUDGET, seed = RPA_SEED)
    pi_ = QNTT.pd_probe(H, x; part = :ii, budget = PROBE_BUDGET, seed = RPA_SEED)
    for (part, p) in ((":rr  (A + B)", pr), (":ii  (A − B)", pi_))
        @printf("  %-8s pd_probe %s  %-17s λ = %+.3e   residual = %-9s  %3d applies\n",
                label, part, p.verdict, p.lambda, isnan(p.residual) ? "n/a" : @sprintf("%.1e", p.residual), p.applies)
    end
    pr.is_pd === false && println("  $label   ‖ A + B has a negative witness: this is NOT a minimizer of the real energy")
    pi_.is_pd === false && println("  $label   ‖ A − B has a negative witness: complex-unstable real state — " *
                                   "the solve below reports it as a counting offset")
    empty = (; ω = Float64[], f_w = Float64[], resonances = Float64[], metric = :none, converged = false,
               iterations = 0, applies = 0, negatives_certified = true, pr, pi = pi_)
    res = nothing
    message = ""
    try
        res = QNTT.rpa_signed(H, x; k = RPA_K, tol = RPA_TOL, max_iter = RPA_MAX_ITER,
                              seed = RPA_SEED)
    catch err
        message = sprint(showerror, err)
        println("  $label   rpa_signed refused: $message")
    end
    res === nothing && return (; empty..., message, wall = time() - t0)
    ω2, info = res
    pos = ω2 .> 0
    ω, f_w, resonances = sqrt.(ω2[pos]), info.f_w[pos], sqrt.(-ω2[.!pos])
    @printf("  %-8s rpa_signed (%s kernel) — %d excitations, %d resonances%s; converged = %s, max residual %.1e, %d basis vectors, %d applies, %.1f s\n",
            label, info.kernel,
            length(ω), length(resonances),
            isempty(resonances) ? "" : (info.negatives_certified ? " (all certified)" : " (NOT all certified — do not read the iκ)"),
            info.converged, maximum(info.residuals; init = 0.0), info.iterations, info.applies, time() - t0)
    offset = something(info.report.neg_apb, 0) + something(info.report.neg_amb, 0)
    events = something(info.report.metric_events, 0)
    (offset > 0 || events > 0) && @printf("  %-8s ‖ indefinite point (counting offset %d, %d metric events, %d de-excitations, %d complex modes): `converged` does not license these ω\n",
                                          label, offset, events, count(<(0), info.krein), length(info.complex))
    isempty(resonances) || @printf("  %-8s resonances iκ, κ = %s\n", label, fmt_ω(resonances, RPA_K))
    flush(stdout)
    return (; ω, f_w, resonances, metric = info.metric, info.converged, info.iterations, info.applies,
              info.negatives_certified, pr, pi = pi_, message, wall = time() - t0)
end

R = Dict{String,Any}()
for dir in RUN
    s = S[dir]
    println(s.sys.label)
    # `rpasigned`: results of the never-refusing signed solver; a cache entry from the refusing
    # one (tag `rpa`) is not reused.
    tag_c1 = @sprintf("%s_%s_rpasigned_c1_eps%.0e_cap%d_k%d", dir, LAYER, DMRG_EPS, DMRG_CAP, RPA_K)
    tag_pg = @sprintf("%s_%s_rpasigned_pg_eps%.0e_cap%d_k%d", dir, LAYER, DMRG_EPS, DMRG_CAP, RPA_K)
    r_c1, how, t = cached_value(tag_c1, () -> response(s.H_c1, P[dir].c1, "untyped"))
    report_cache(tag_c1, how, t); flush(stdout)
    r_pg, how, t = cached_value(tag_pg, () -> response(s.H_pg, P[dir].pg, "typed"))
    report_cache(tag_pg, how, t); flush(stdout)
    R[dir] = (c1 = r_c1, pg = r_pg)
    for (label, r) in (("untyped", r_c1), ("typed A1", r_pg))
        @printf("  %-9s ω[1:%d]   = %s%s\n", label, RPA_K, fmt_ω(r.ω, RPA_K),
                isempty(r.ω) ? "" : (r.converged ? "" : "   (NOT at tolerance — lowest values settle first)"))
        @printf("  %-9s f_W[1:%d] = %s\n", label, RPA_K, fmt_fw(r.f_w, RPA_K))
    end
end

water CAS(8,6) / 6-31G
  untyped  pd_probe :rr  (A + B)  inconclusive      λ = +4.652e+00   residual = 1.6e+00    105 applies
  untyped  pd_probe :ii  (A − B)  inconclusive      λ = +5.135e+00   residual = 2.1e+00    105 applies


  untyped  rpa_signed (paired kernel) — 4 excitations, 2 resonances (all certified); converged = false, max residual 1.4e-03, 1200 basis vectors, 1204 applies, 3347.6 s
  untyped  ‖ indefinite point (counting offset 8, 0 metric events, 0 de-excitations, 3 complex modes): `converged` does not license these ω


  untyped  resonances iκ, κ = 1.72822 1.58164


cache MISS h2o_631g_su2_rpasigned_c1_eps1e-03_cap256_k4 — built in 3349.5 s and saved for next time


  typed    pd_probe :rr  (A + B)  inconclusive      λ = +4.412e+00   residual = 2.5e+00    105 applies
  typed    pd_probe :ii  (A − B)  inconclusive      λ = +4.210e+00   residual = 2.2e+00    105 applies


  typed    rpa_signed (paired kernel) — 4 excitations, 0 resonances; converged = true, max residual 7.9e-07, 474 basis vectors, 474 applies, 452.8 s


... (13 outputs left out)

  typed    pd_probe :rr  (A + B)  inconclusive      λ = +1.383e+00   residual = 1.3e+00    105 applies
  typed    pd_probe :ii  (A − B)  inconclusive      λ = +1.802e+00   residual = 9.1e-01    105 applies


  typed    rpa_signed (paired kernel) — 4 excitations, 0 resonances; converged = true, max residual 5.9e-07, 934 basis vectors, 934 applies, 4499.2 s


cache MISS c2h4_631g_su2_rpasigned_pg_eps1e-03_cap256_k4 — built in 4499.3 s and saved for next time


  untyped   ω[1:4]   = 0.29150 0.30869 0.34259 0.35635   (NOT at tolerance — lowest values settle first)
  untyped   f_W[1:4] = 1.00    0.82    0.11    0.07
  typed A1  ω[1:4]   = 0.52131 0.55700 0.59993 0.65490
  typed A1  f_W[1:4] = 0.03    0.19    0.90    0.05


### Table

Rows: the paper's two LR values (SS and SA printed alongside) and, for water, the FCI gaps.
Columns: the nearest mode of each method with its index, deviation and `f_W`. `—` where no
spectrum was published.

In [8]:
for dir in RUN
    s, b, r = S[dir], B[dir], R[dir]
    println("\n", s.sys.label, "   (ω in Hartree; entry = nearest mode [index] deviation)")
    @printf("  %-26s %-34s %-34s %-34s\n", "reference", "CASSCF-LR (ours)", "DMRG untyped TT-RPA",
            "DMRG typed TT-RPA (A1 only)")
    keep = b.converged && b.status in (:stable, :resonant) ? (b.ω .> 0) : falses(length(b.ω))
    ω_cas, fw_cas = b.ω[keep], b.f_w[keep]
    rows = [(@sprintf("paper LR ω%d = %.6f", j, PAPER[s.sys.mol][j, 3]), PAPER[s.sys.mol][j, 3]) for j in 1:2]
    if s.fci_gaps !== nothing
        append!(rows, [(@sprintf("FCI gap %d   = %.6f", j, g), g) for (j, g) in enumerate(s.fci_gaps)])
    end
    for (name, g) in rows
        @printf("  %-26s %s  %s  %s\n", name, fmt_near(nearest(ω_cas, g), fw_cas),
                fmt_near(nearest(r.c1.ω, g), r.c1.f_w), fmt_near(nearest(r.pg.ω, g), r.pg.f_w))
    end
    @printf("  paper SS / SA for context: ω₁ = %.6f / %.6f   ω₂ = %.6f / %.6f\n",
            PAPER[s.sys.mol][1, 1], PAPER[s.sys.mol][1, 2], PAPER[s.sys.mol][2, 1], PAPER[s.sys.mol][2, 2])
    @printf("  full lists: CASSCF-LR %s | untyped %s | typed %s\n",
            fmt_ω(ω_cas, RPA_K), fmt_ω(r.c1.ω, RPA_K), fmt_ω(r.pg.ω, RPA_K))
    for (label, r) in (("untyped", r.c1), ("typed", r.pg))
        r.metric === :rr && @printf("  %s: H^ii indefinite at this real state — metric swapped to H^rr; resonances iκ, κ = %s (%s)\n",
                                    label, fmt_ω(r.resonances, RPA_K), r.negatives_certified ? "certified" : "NOT certified")
    end
end


water CAS(8,6) / 6-31G   (ω in Hartree; entry = nearest mode [index] deviation)
  reference                  CASSCF-LR (ours)                   DMRG untyped TT-RPA                DMRG typed TT-RPA (A1 only)       
  paper LR ω1 = 0.313274     0.313274 [ 1] +1.5e-07 f_W  0.10  0.294860 [ 1] -1.8e-02 f_W  0.00  0.390130 [ 1] +7.7e-02 f_W  0.00


  paper LR ω2 = 0.388773     0.388773 [ 2] +3.5e-07 f_W  0.06  0.390132 [ 3] +1.4e-03 f_W  0.00  0.390130 [ 1] +1.4e-03 f_W  0.00
  FCI gap 1   = 0.266970     0.313274 [ 1] +4.6e-02 f_W  0.10  0.294860 [ 1] +2.8e-02 f_W  0.00  0.390130 [ 1] +1.2e-01 f_W  0.00
  FCI gap 2   = 0.294410     0.313274 [ 1] +1.9e-02 f_W  0.10  0.294860 [ 1] +4.5e-04 f_W  0.00  0.390130 [ 1] +9.6e-02 f_W  0.00
  paper SS / SA for context: ω₁ = 0.299801 / 0.279149   ω₂ = 0.377632 / 0.357262
  full lists: CASSCF-LR 0.31327 0.38877 0.41295 0.49290 | untyped 0.29486 0.37379 0.39013 0.47373 | typed 0.39013 0.67213 0.74824 0.89408



formaldehyde CAS(4,3) / 6-31G   (ω in Hartree; entry = nearest mode [index] deviation)
  reference                  CASSCF-LR (ours)                   DMRG untyped TT-RPA                DMRG typed TT-RPA (A1 only)       
  paper LR ω1 = 0.138848     0.137479 [ 1] -1.4e-03 f_W  0.16  0.154268 [ 2] +1.5e-02 f_W  0.82  0.368416 [ 1] +2.3e-01 f_W  0.97
  paper LR ω2 = 0.325269     0.320439 [ 2] -4.8e-03 f_W  0.09  0.356850 [ 4] +3.2e-02 f_W  0.99  0.368416 [ 1] +4.3e-02 f_W  0.97
  paper SS / SA for context: ω₁ = 0.144960 / 0.147317   ω₂ = 0.335188 / 0.464643
  full lists: CASSCF-LR 0.13748 0.32044 0.39396 0.39399 | untyped 0.06079 0.15427 0.17466 0.35685 | typed 0.36842 0.38209 0.41833 0.48008

ethylene CAS(2,2) / 6-31G   (ω in Hartree; entry = nearest mode [index] deviation)
  reference                  CASSCF-LR (ours)                   DMRG untyped TT-RPA                DMRG typed TT-RPA (A1 only)       
  paper LR ω1 = 0.354318     0.354318 [ 2] +1.8e-07 f_W  0.07  0.356350 [ 4] +2.0e

## 5. GAD

`saddle_seed` kicks the polished state by `GAD_KICK` along the softest `A + B` mode (radial mode
deflated). `gad` runs gentlest-ascent dynamics with the damped Weingarten model, grid preserved,
and certifies the index at the endpoint with the exact `A + B`. Reported: endpoint energy above
the ground state beside the nearest RPA ω of the same arm, `‖grad‖`, the index certificate,
steps, rejections, applies, `eig1`, `b_gain_max`, `min_schmidt_min`, wall time. On the typed
state only A1 / Ag saddles are reachable. Long GAD trajectories can end at different saddles on
different CPU architectures. `GAD_ON_CASSCF = true` also launches from the CASSCF point.

In [9]:
const GAD_KICK         = 5e-2    # step along the softest A+B mode
const GAD_KICK_BUDGET  = 100     # Davidson budget for that mode
const GAD_STEPS        = 100     # accepted-step budget
const GAD_GRADTOL      = 1e-7
const GAD_PROBE_BUDGET = 100     # the index-1 certificate's Davidson budget
const GAD_SEED         = 0x6AD
const GAD_ON_CASSCF    = false   # also launch from the CASSCF point (re-runs the SCF; uncached)

function gad_demo(H, x, label)
    t0 = time()
    λ_gs = QNTT.rayleigh_quotient(H, x)
    xk, λ_witness = QNTT.saddle_seed(H, x; eps = GAD_KICK, budget = GAD_KICK_BUDGET, seed = GAD_SEED)
    g0 = QNTT.grad_norm(H, xk)
    @printf("  %-8s kick: softest A+B eigenvalue %.4f, ‖grad‖ after kick %.2e\n", label, λ_witness, g0)
    xe, info = QNTT.gad(H, xk; gradtol = GAD_GRADTOL, max_steps = GAD_STEPS, k_davidson = 8,
                        w_model = :damped, certify = true, probe_budget = GAD_PROBE_BUDGET,
                        preserve_grid = true, seed = GAD_SEED)
    λe = QNTT.rayleigh_quotient(H, xe)
    flush(stdout)
    return (; λ_gs, λ_witness, g0, λe, gap = λe - λ_gs, info..., wall = time() - t0)
end

function report_gad(g, label, ω_ref, paper_ω1)
    status = g.converged ? (g.index1_certified ? "converged, index-1 CERTIFIED" :
                            @sprintf("converged, index estimate %s (not certified)", string(g.index_estimate))) :
                           @sprintf("NOT converged in %d steps (‖grad‖ = %.2e)", g.steps, g.gnorm)
    @printf("  %-8s %s\n", label, status)
    @printf("  %-8s endpoint λ − λ_gs = %.6f   nearest RPA ω of this arm: %s   paper LR ω₁ = %.6f\n",
            label, g.gap, fmt_near(nearest(ω_ref, g.gap)), paper_ω1)
    @printf("  %-8s %d accepted / %d rejected steps, %d relatches, %d Hessian applies, eig1 = %s, b_gain_max = %.2e, min σ = %.2e, %.1f s\n",
            label, g.steps, g.rejected_steps, g.relatches, g.applies,
            g.eig1 === nothing ? "n/a" : @sprintf("%.4f", g.eig1), g.b_gain_max, g.min_schmidt_min, g.wall)
end

G = Dict{String,Any}()
for dir in RUN
    s = S[dir]
    println(s.sys.label)
    tag_c1 = @sprintf("%s_%s_gad_c1_eps%.0e_cap%d", dir, LAYER, DMRG_EPS, DMRG_CAP)
    tag_pg = @sprintf("%s_%s_gad_pg_eps%.0e_cap%d", dir, LAYER, DMRG_EPS, DMRG_CAP)
    g_c1, how, t = cached_value(tag_c1, () -> gad_demo(s.H_c1, P[dir].c1, "untyped"))
    report_cache(tag_c1, how, t); flush(stdout)
    g_pg, how, t = cached_value(tag_pg, () -> gad_demo(s.H_pg, P[dir].pg, "typed"))
    report_cache(tag_pg, how, t); flush(stdout)
    G[dir] = (c1 = g_c1, pg = g_pg)
    report_gad(g_c1, "untyped", R[dir].c1.ω, PAPER[s.sys.mol][1, 3])
    report_gad(g_pg, "typed", R[dir].pg.ω, PAPER[s.sys.mol][1, 3])
    if GAD_ON_CASSCF
        # The CASSCF point re-converged here (the baseline cell cached numbers, not the rotated operator).
        λ0, x0 = QNTT.cas_reference_state(s.H_c1, s.sys.cas; m_active = 0, seed = CASSCF_SEED)
        λ, x, u, H_rot, info = QNTT.casscf(s.data, x0, s.sys.cas; orb_max_iter = CASSCF_ORB_ITER,
                                           orb_grad = CASSCF_ORB_GRAD, tt_grad_tol = CASSCF_TT_GRAD,
                                           preserve_grid = true)
        info.converged || (println("  casscf   SCF did not certify — GAD skipped"); continue)
        g_cas = gad_demo(H_rot, x, "casscf")
        report_gad(g_cas, "casscf", B[dir].ω, PAPER[s.sys.mol][1, 3])
    end
end

water CAS(8,6) / 6-31G
cache HIT  h2o_631g_su2_gad_c1_eps1e-03_cap256 — loaded in 0.000 s (the pipeline below was skipped)


cache HIT  h2o_631g_su2_gad_pg_eps1e-03_cap256 — loaded in 0.000 s (the pipeline below was skipped)


  untyped  NOT converged in 100 steps (‖grad‖ = 1.37e-04)
  untyped  endpoint λ − λ_gs = -0.000000   nearest RPA ω of this arm: 0.294860 [ 1] +2.9e-01   paper LR ω₁ = 0.313274


  untyped  100 accepted / 200 rejected steps, 4 relatches, 724 Hessian applies, eig1 = n/a, b_gain_max = 1.46e-01, min σ = 1.89e-05, 1423.4 s
  typed    NOT converged in 100 steps (‖grad‖ = 3.89e-04)


  typed    endpoint λ − λ_gs = 0.000000   nearest RPA ω of this arm: 0.390130 [ 1] +3.9e-01   paper LR ω₁ = 0.313274
  typed    100 accepted / 64 rejected steps, 0 relatches, 709 Hessian applies, eig1 = n/a, b_gain_max = 1.58e-01, min σ = 7.11e-05, 419.1 s
formaldehyde CAS(4,3) / 6-31G
cache HIT  hcho_631g_su2_gad_c1_eps1e-03_cap256 — loaded in 0.000 s (the pipeline below was skipped)


cache HIT  hcho_631g_su2_gad_pg_eps1e-03_cap256 — loaded in 0.000 s (the pipeline below was skipped)


  untyped  NOT converged in 100 steps (‖grad‖ = 3.41e-04)
  untyped  endpoint λ − λ_gs = 0.000001   nearest RPA ω of this arm: 0.060787 [ 1] +6.1e-02   paper LR ω₁ = 0.138848
  untyped  100 accepted / 63 rejected steps, 0 relatches, 1473 Hessian applies, eig1 = n/a, b_gain_max = 1.07e+00, min σ = 1.90e-05, 15956.2 s
  typed    NOT converged in 100 steps (‖grad‖ = 1.50e-04)
  typed    endpoint λ − λ_gs = 0.000000   nearest RPA ω of this arm: 0.368416 [ 1] +3.7e-01   paper LR ω₁ = 0.138848
  typed    100 accepted / 63 rejected steps, 0 relatches, 1246 Hessian applies, eig1 = n/a, b_gain_max = 9.82e-01, min σ = 2.29e-05, 4111.5 s
ethylene CAS(2,2) / 6-31G
cache HIT  c2h4_631g_su2_gad_c1_eps1e-03_cap256 — loaded in 0.000 s (the pipeline below was skipped)


cache HIT  c2h4_631g_su2_gad_pg_eps1e-03_cap256 — loaded in 0.000 s (the pipeline below was skipped)


  untyped  NOT converged in 100 steps (‖grad‖ = 5.29e-04)
  untyped  endpoint λ − λ_gs = 0.000000   nearest RPA ω of this arm: 0.291505 [ 1] +2.9e-01   paper LR ω₁ = 0.354318
  untyped  100 accepted / 62 rejected steps, 0 relatches, 1496 Hessian applies, eig1 = n/a, b_gain_max = 1.03e+00, min σ = 4.73e-05, 24453.3 s
  typed    NOT converged in 100 steps (‖grad‖ = 4.63e-03)
  typed    endpoint λ − λ_gs = 0.000039   nearest RPA ω of this arm: 0.521309 [ 1] +5.2e-01   paper LR ω₁ = 0.354318
  typed    100 accepted / 61 rejected steps, 0 relatches, 1450 Hessian applies, eig1 = n/a, b_gain_max = 9.18e-01, min σ = 3.81e-05, 5339.4 s


## Summary

In [10]:
for dir in RUN
    s, b = S[dir], B[dir]
    println("\n", s.sys.label)
    @printf("  %-34s %.9f\n", "E_HF (fixture)", s.fx.e_hf)
    @printf("  %-34s %.9f\n", "CASSCF (pyscf, reference file)", s.casscf_gs)
    @printf("  %-34s %.9f   %s\n", "CASSCF-equivalent (ours)", b.E, b.converged ? "certified" : "NOT certified")
    for (label, H, y) in (("DMRG untyped, polished", s.H_c1, P[dir].c1), ("DMRG typed, polished", s.H_pg, P[dir].pg))
        @printf("  %-34s %.9f   pooled ranks %s\n", label, QNTT.rayleigh_quotient(H, y) + s.fx.e_nuc,
                string([sum(g) for g in QNTT.ranks(y)]))
    end
    isnan(s.fx.e_ref) || @printf("  %-34s %.9f\n", s.fci_gaps === nothing ? "E_ref (CCSD(T))" : "E_ref (FCI)", s.fx.e_ref)
    @printf("  %-34s %s\n", "ω: CASSCF-LR", fmt_ω(b.converged && b.status in (:stable, :resonant) ? b.ω : Float64[], RPA_K))
    @printf("  %-34s %s\n", "ω: DMRG untyped TT-RPA", fmt_ω(R[dir].c1.ω, RPA_K))
    @printf("  %-34s %s\n", "ω: DMRG typed TT-RPA (A1)", fmt_ω(R[dir].pg.ω, RPA_K))
    @printf("  %-34s %.6f / %.6f\n", "GAD gap untyped / typed", G[dir].c1.gap, G[dir].pg.gap)
end


water CAS(8,6) / 6-31G
  E_HF (fixture)                     -75.980157922
  CASSCF (pyscf, reference file)     -76.039557117
  CASSCF-equivalent (ours)           -76.039557119   certified


  DMRG untyped, polished             -76.121256538   pooled ranks [1, 3, 7, 19, 49, 143, 256, 256, 196, 104, 31, 10, 3, 1]
  DMRG typed, polished               -76.121257687   pooled ranks [1, 3, 7, 19, 50, 144, 256, 256, 196, 104, 31, 10, 3, 1]
  E_ref (FCI)                        -76.121285071
  ω: CASSCF-LR                       0.31327 0.38877 0.41295 0.49290
  ω: DMRG untyped TT-RPA             0.29486 0.37379 0.39013 0.47373
  ω: DMRG typed TT-RPA (A1)          0.39013 0.67213 0.74824 0.89408
  GAD gap untyped / typed            -0.000000 / 0.000000



formaldehyde CAS(4,3) / 6-31G
  E_HF (fixture)                     -113.806879119
  CASSCF (pyscf, reference file)     -113.849424553
  CASSCF-equivalent (ours)           -113.847644511   certified
  DMRG untyped, polished             -114.041442914   pooled ranks [1, 3, 5, 12, 24, 59, 133, 256, 256, 256, 256, 256, 256, 256, 255, 217, 163, 118, 67, 30, 10, 3, 1]


  DMRG typed, polished               -114.041325376   pooled ranks [1, 3, 5, 12, 24, 59, 132, 256, 256, 256, 256, 256, 256, 256, 254, 215, 162, 117, 67, 30, 10, 3, 1]
  E_ref (CCSD(T))                    -114.043672444
  ω: CASSCF-LR                       -2.08294 -1.02912 0.13748 0.32044
  ω: DMRG untyped TT-RPA             0.06079 0.15427 0.17466 0.35685
  ω: DMRG typed TT-RPA (A1)          0.36842 0.38209 0.41833 0.48008
  GAD gap untyped / typed            0.000001 / 0.000000

ethylene CAS(2,2) / 6-31G
  E_HF (fixture)                     -78.004104910
  CASSCF (pyscf, reference file)     -78.034920730
  CASSCF-equivalent (ours)           -78.034920730   certified
  DMRG untyped, polished             -78.214331986   pooled ranks [1, 3, 5, 13, 26, 62, 133, 202, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 164, 145, 112, 84, 49, 23, 10, 3, 1]
  DMRG typed, polished               -78.214216117   pooled ranks [1, 3, 5, 13, 26, 61, 129, 196, 256, 256, 256, 256, 256, 256, 256, 256, 

  E_ref (CCSD(T))                    -78.218487193
  ω: CASSCF-LR                       0.35162 0.35432 0.37810 0.38916
  ω: DMRG untyped TT-RPA             0.29150 0.30869 0.34259 0.35635
  ω: DMRG typed TT-RPA (A1)          0.52131 0.55700 0.59993 0.65490
  GAD gap untyped / typed            0.000000 / 0.000039


- ω are fixed-rank values at the ranks printed beside them, in the fixture orbitals.
- The typed column is A1 / Ag-only.
- Ranks on this layer are the reduced multiplet rank (NOT comparable to an abelian pooled M).